In [27]:
import numpy as np
import pandas as pd
import stumpy as st
import datetime
import random
import string
from scipy.stats import pearsonr


## Load Stage
human_freedom_main_index_path = r"/workspace/qgi\trojan\data bank\Human and Economic Freedom\CATO Institute\MAIN - Human and Economic Freedom.csv"
df                            = pd.read_csv(human_freedom_main_index_path)
df                            = df[["Year", "Country", "HUMAN FREEDOM"]]
index_column_name             = "HUMAN FREEDOM"
correlation_threshold         = 0.5
pattern_lengths_list          = [3, 4, 5, 6, 7, 8, 9, 10]
country_A                     = "Germany"
country_B                     = "France"


## Functions pool
def remove_nan_rows(df: pd.DataFrame, index_name: str) -> pd.DataFrame:
    """Utility function to remove rows with NaN values in the specified index column."""
    return df.dropna(subset=[index_name])


def interpolate_single_missing_value(df: pd.DataFrame, index_name: str) -> pd.DataFrame:
    """
    Interpolate a single missing value for each country's time series in the dataframe.
    
    The function identifies and interpolates only one missing value per country based on two conditions:
    1. The missing value should have actual numerical values both before and after it.
    2. Interpolating this missing value should result in the longest possible consecutive sequence 
       of numerical observations within the country's time series.
    
    Parameters:
    - df (pd.DataFrame): The input dataframe containing columns "Year", "Country", and the specified index name.
    - index_name (str): The name of the column in the dataframe that contains the time series data to be interpolated.
    
    Returns:
    - pd.DataFrame: The dataframe with the interpolated value(s).
    
    Note:
    The function uses linear interpolation to estimate the missing value.
    """
    countries = df["Country"].unique()
    for country in countries:
        country_df = df[df["Country"] == country].copy()
        country_df.set_index("Year", inplace=True)
        
        # Find missing values
        missing_years = country_df[country_df[index_name].isna()].index.tolist()
        
        # If there are no missing values or only one missing value, continue to next country
        if len(missing_years) <= 1:
            continue
        
        # Find the best missing year to interpolate based on the conditions
        best_missing_year = None
        best_sequence_length = 0
        
        for year in missing_years:
            if year - 1 in country_df.index and year + 1 in country_df.index:
                if not np.isnan(country_df.at[year - 1, index_name]) and not np.isnan(country_df.at[year + 1, index_name]):
                    # Calculate sequence length if this year is interpolated
                    sequence_length = 2  # Including the year before and after
                    y = year + 1
                    while y + 1 in country_df.index and not np.isnan(country_df.at[y + 1, index_name]):
                        sequence_length += 1
                        y += 1
                    y = year - 1
                    while y - 1 in country_df.index and not np.isnan(country_df.at[y - 1, index_name]):
                        sequence_length += 1
                        y -= 1
                    
                    # Update best year if this year produces a longer sequence
                    if sequence_length > best_sequence_length:
                        best_sequence_length = sequence_length
                        best_missing_year = year
        
        # Interpolate the best missing year
        if best_missing_year:
            interpolated_value = (country_df.at[best_missing_year - 1, index_name] + country_df.at[best_missing_year + 1, index_name]) / 2
            country_df.at[best_missing_year, index_name] = interpolated_value
            df.update(country_df)
    
    return df


def generate_unique_id(country_A: str, country_B: str, pattern_length: int, index_name: str) -> str:
    """Generate a unique ID based on the provided parameters."""
    random_chars = ''.join(random.choices(string.ascii_uppercase + string.digits, k=8))
    return f"{country_A}_{country_B}_{pattern_length}_{index_name}_{random_chars}"


## Transformation Stage
## filter for countries that have at least 3 consecutive values
df              = df.sort_values(by = "Year")                             # sorting by year
df_interpolated = interpolate_single_missing_value(df, index_column_name) # interpolate NaNs
df_cleaned      = remove_nan_rows(df_interpolated, index_column_name)     # removing NaNs


## Patternize Stage
def find_patterns(df: pd.DataFrame, correlation_threshold: float, index_name: str, country_A = None, country_B = None, pattern_lengths_list = None):
    results = [] # Store results
    
    # If `pattern_lengths_list`` is not provided, generate it based on the range of years in the dataset minus one. 
    if pattern_lengths_list is None:
        min_year = df["Year"].min()
        max_year = df["Year"].max()
        pattern_lengths_list = list(range(3, max_year - min_year + 1))
    
    if min(pattern_lengths_list) < 3:
        # Ensure pattern_length is at least 3
        raise ValueError(f"Pattern length {min(pattern_lengths_list)} is less than 3 years.")
    
    countries_in_df = df["Country"].unique().tolist() # List of countries in the dataframe
    
    # Validate country_A
    if country_A:
        if country_A not in countries_in_df:
            raise ValueError(f"country_A '{country_A}' not found in the dataset.")
        countries_to_process = [country_A]
    else:
        countries_to_process = countries_in_df
    
    # Validate country_B
    if country_B:
        if country_B not in countries_in_df:
            raise ValueError(f"country_B '{country_B}' not found in the dataset.")
        countries_to_compare = [country_B]
    else:
        countries_to_compare = countries_in_df
    
    for country in countries_to_process:
        for pattern_length in pattern_lengths_list:
            country_ts = df[df["Country"] == country][index_name].values # Extract the time series for the current country
        
            if pattern_length > len(country_ts): # Check if the pattern_length is valid for the current country's time series
                print(f"Skipping pattern length {pattern_length} for country '{country}' as its time series length is {len(country_ts)}. ")
                continue
            
            matrix_profile = st.stump(country_ts, m = pattern_length)           # Compute the matrix profile for the current country using stumpy
            from_index     = np.argmin(matrix_profile[:, 0])                    # Find the starting index of the most similar pattern in the current country's time series
            pattern        = country_ts[from_index:from_index + pattern_length] # Extract the pattern from the current country's time series
            for other_country in countries_to_compare:                          # Iterate over countries_to_compare to find patterns
                if other_country == country:
                    continue
                other_country_df = df[df["Country"] == other_country]
                other_country_ts = other_country_df[index_name].values
                for i in range(len(other_country_ts) - pattern_length + 1):
                    other_pattern = other_country_ts[i: i + pattern_length]
                    corr, _       = pearsonr(pattern, other_pattern)
                    if corr >= correlation_threshold:
                        unique_id = generate_unique_id(country, other_country, pattern_length, index_name)
                        results.append({
                            "Index Name":     index_name,
                            "Country A":      country,
                            "Country B":      other_country,
                            "Pattern Length": pattern_length,
                            "Correlation":    corr,
                            "Start Year A":   df[df["Country"] == country]["Year"].iloc[from_index],
                            "End Year A":     df[df["Country"] == country]["Year"].iloc[from_index + pattern_length - 1],
                            "Index Values A": list(pattern),
                            "Start Year B":   other_country_df["Year"].iloc[i],
                            "End Year B":     other_country_df["Year"].iloc[i + pattern_length - 1],
                            "Index Values B": list(other_pattern),
                            "Creation Date":  datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                            "Unique ID":      unique_id
                        })

    return pd.DataFrame(results)


## Example usage:
# df_cleaned is your cleaned dataframe with columns: Year, Country, and the specified index name
#results = find_patterns(country_A, df_cleaned, correlation_threshold, index_column_name, country_B, None)
results = find_patterns(df_cleaned, correlation_threshold, index_column_name, None, None, None)

/workspace\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\scipy\stats\_stats_py.py:4427: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


Skipping pattern length 20 for country 'Georgia' as its time series length is 19. 
Skipping pattern length 20 for country 'Vietnam' as its time series length is 19. 
Skipping pattern length 19 for country 'Mozambique' as its time series length is 18. 
Skipping pattern length 20 for country 'Mozambique' as its time series length is 18. 
Skipping pattern length 19 for country 'North Macedonia' as its time series length is 18. 
Skipping pattern length 20 for country 'North Macedonia' as its time series length is 18. 
Skipping pattern length 18 for country 'Azerbaijan' as its time series length is 17. 
Skipping pattern length 19 for country 'Azerbaijan' as its time series length is 17. 
Skipping pattern length 20 for country 'Azerbaijan' as its time series length is 17. 
Skipping pattern length 18 for country 'Armenia' as its time series length is 17. 
Skipping pattern length 19 for country 'Armenia' as its time series length is 17. 
Skipping pattern length 20 for country 'Armenia' as its 

In [28]:
results # 1073787
        # 1069947

,Index Name,Country A,Country B,Pattern Length,Correlation,Start Year A,End Year A,Index Values A,Start Year B,End Year B,Index Values B,Pattern Creation Date,Unique ID
0,HUMAN FREEDOM,Zimbabwe,Germany,3,0.662849,2004,2006,"[4.64, 4.62, 4.76]",2001,2003,"[8.63, 8.67, 8.69]",2023-10-24 01:49:47,Zimbabwe_Germany_3_HUMAN FREEDOM_B6524YD0
1,HUMAN FREEDOM,Zimbabwe,Germany,3,0.979864,2004,2006,"[4.64, 4.62, 4.76]",2007,2009,"[8.62, 8.61, 8.64]",2023-10-24 01:49:47,Zimbabwe_Germany_3_HUMAN FREEDOM_BRJT8MNI
2,HUMAN FREEDOM,Zimbabwe,Germany,3,0.717204,2004,2006,"[4.64, 4.62, 4.76]",2008,2010,"[8.61, 8.64, 8.66]",2023-10-24 01:49:47,Zimbabwe_Germany_3_HUMAN FREEDOM_O9C3PLBN
3,HUMAN FREEDOM,Zimbabwe,Germany,3,0.893405,2004,2006,"[4.64, 4.62, 4.76]",2009,2011,"[8.64, 8.66, 8.7]",2023-10-24 01:49:47,Zimbabwe_Germany_3_HUMAN FREEDOM_NPUY7DCH
4,HUMAN FREEDOM,Zimbabwe,Germany,3,0.962103,2004,2006,"[4.64, 4.62, 4.76]",2011,2013,"[8.7, 8.68, 8.73]",2023-10-24 01:49:47,Zimbabwe_Germany_3_HUMAN FREEDOM_E3HJWBWS
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1069942,HUMAN FREEDOM,Sudan,Bhutan,5,0.728714,2016,2020,"[3.7, 3.61, 3.46, 4.04, 4.31]",2016,2020,"[6.61, 6.53, 6.64, 6.68, 6.71]",2023-10-24 02:11:31,Sudan_Bhutan_5_HUMAN FREEDOM_EW5ZAI9M
1069943,HUMAN FREEDOM,Sudan,Seychelles,5,0.612593,2016,2020,"[3.7, 3.61, 3.46, 4.04, 4.31]",2014,2018,"[7.57, 7.84, 7.72, 7.86, 7.91]",2023-10-24 02:11:31,Sudan_Seychelles_5_HUMAN FREEDOM_I0ZO83GB
1069944,HUMAN FREEDOM,Sudan,Seychelles,5,0.627908,2016,2020,"[3.7, 3.61, 3.46, 4.04, 4.31]",2015,2019,"[7.84, 7.72, 7.86, 7.91, 7.91]",2023-10-24 02:11:31,Sudan_Seychelles_5_HUMAN FREEDOM_EBY3T4BX
1069945,HUMAN FREEDOM,Sudan,Lao PDR,5,0.840703,2016,2020,"[3.7, 3.61, 3.46, 4.04, 4.31]",2014,2018,"[5.39, 5.27, 5.42, 5.55, 5.61]",2023-10-24 02:11:32,Sudan_Lao PDR_5_HUMAN FREEDOM_QUZW51YQ
